# Automatic Evaluation: quality

Chrf, BLEU, COMET-22


In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="5"

In [ ]:
def read_file(fname):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    return output

In [ ]:
def get_translations(df, lp, context_type, model_name, split="dev"):
  source_lang = lp.split("-")[0]
  target_lang = lp.split("-")[1].replace('pt','pt-br')

  xx_yy_generations = read_file(f"generations/{context_type}/mt/wmt24_chat_{split}.{source_lang}-{target_lang}/vllm/{model_name}/generation.txt")
  yy_xx_generations = read_file(f"generations/{context_type}/mt/wmt24_chat_{split}.{target_lang}-{source_lang}/vllm/{model_name}/generation.txt")

  assert len(xx_yy_generations) + len(yy_xx_generations) == len(df)
  translations = []
  j,k=0,0

  for i, row in df.iterrows():
    if row["lp"] == f"{source_lang}-{target_lang}":
      translations.append(xx_yy_generations[j])
      j+=1
    else:
      # nl-en and it-en are considered here
      translations.append(yy_xx_generations[k])
      k+=1
  return translations

In [ ]:
import sacrebleu
import numpy as np
from comet import download_model, load_from_checkpoint
ref_metric = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))
qe_metric = load_from_checkpoint(download_model("Unbabel/wmt20-comet-qe-da"))

def get_scores(df, col_mt, metric="chrf"):
  if metric=="chrf":
    return [sacrebleu.sentence_chrf(x, [y]).score for (x, y) in zip(df[col_mt].to_list(), df["reference"].to_list())]
  elif metric=="comet":
    scores = ref_metric.predict([{"mt": y, "ref":z, "src": x} for x, y, z in zip(df["source"].to_list(),
                                                                               df[col_mt].to_list(),
                                                                               df["reference"].to_list())],
                              batch_size=256, gpus=1)['scores']
    return scores
  elif metric=="comet-qe":
    scores = qe_metric.predict([{"mt": y, "src": x} for x, y in zip(df["source"].to_list(),
                                                                    df[col_mt].to_list())],
                              batch_size=256, gpus=1)['scores']
    return scores

In [ ]:
import json
def get_stored_scores(df, lp, context_type, model_name, metric_name, split="dev"):
    source_lang = lp.split("-")[0]
    target_lang = lp.split("-")[1].replace('pt','pt-br')
    with open(f'evaluations/{context_type}/mt/wmt24_chat_{split}.{source_lang}-{target_lang}/vllm/{model_name}/evaluation.json', "r",) as f:
        xx_yy_scores = json.load(f)[f"{metric_name}_segments"]
    with open(f'evaluations/{context_type}/mt/wmt24_chat_{split}.{target_lang}-{source_lang}/vllm/{model_name}/evaluation.json', "r",) as f:
        yy_xx_scores = json.load(f)[f"{metric_name}_segments"]

    assert len(xx_yy_scores) + len(yy_xx_scores) == len(df)
    scores = []
    j,k=0,0

    for i, row in df.iterrows():
        if row["lp"] == f"{source_lang}-{target_lang}":
            scores.append(xx_yy_scores[j])
            j+=1
        else:
            # nl-en and it-en are considered here
            scores.append(yy_xx_scores[k])
            k+=1
    return scores

In [ ]:
import pandas as pd

model_names = [
                # 'TowerInstruct-13B-v0.2_temperature_0.9_top_p_0.95',
                # 'TowerInstruct-13B-v0.2_temperature_0.8_top_p_0.9',
                # 'TowerInstruct-13B-v0.2_temperature_0.8_top_p_0.95',
                # 'TowerInstruct-13B-v0.2',
                # 'TowerInstruct-13B-v0.2_temperature_0.7_top_p_0.95',
                # 'TowerInstruct-13B-v0.2_temperature_0.6_top_p_0.95',
                # 'TowerInstruct-13B-v0.2_temperature_0.9_top_p_0.9',
                # 'TowerInstruct-13B-v0.2_temperature_1.0_top_p_0.9',
                # 'TowerInstruct-13B-v0.2_temperature_1.0_top_p_0.95',
                # 'TowerInstruct-13B-v0.2_temperature_0.6_top_p_0.9',
                # 'TowerInstruct-13B-v0.2_temperature_0.7_top_p_0.9',
                # 'TowerInstruct-7B-v0.2',
                'TowerInstruct-7B-w-chat-empty-sys',   
                'TowerInstruct-7B-w-chat-empty-sys_temperature_0.7_top_p_0.95',
                'TowerInstruct-7B-w-chat-empty-sys_temperature_0.6_top_p_0.95',
                'TowerInstruct-7B-w-chat-empty-sys_temperature_0.9_top_p_0.9',
                'TowerInstruct-7B-w-chat-empty-sys_temperature_1.0_top_p_0.9',
                'TowerInstruct-7B-w-chat-empty-sys_temperature_1.0_top_p_0.95',
                'TowerInstruct-7B-w-chat-empty-sys_temperature_0.6_top_p_0.9',
                'TowerInstruct-7B-w-chat-empty-sys_temperature_0.7_top_p_0.9',
                'TowerInstruct-7B-w-chat-empty-sys_temperature_0.9_top_p_0.95',
                'TowerInstruct-7B-w-chat-empty-sys_temperature_0.8_top_p_0.9',
                'TowerInstruct-7B-w-chat-empty-sys_temperature_0.8_top_p_0.95',
                ]

for lang_pair in [ "en-de", "en-fr", "en-pt", "en-ko" ,  "en-nl",]:
  df = pd.read_csv(f"chat-task-2024-data/valid/{lang_pair}.csv",
                  keep_default_na=False,
        na_values=["NaN"],)
  df["lp"] = df["source_language"] + "-" + df["target_language"]
  df["length"] = df["source"].apply(lambda x: len(x))

  for model_name in model_names:
    for context_type in ["no_context_empty_sys", "full_context_empty_sys"]:
      df[model_name + "#" + context_type] = get_translations(df, lang_pair, context_type, model_name)

      for metric_name in ["chrf", "comet", "comet_kiwi", "comet-qe"]:
        if metric_name!="comet-qe":
          df[model_name + "#" + context_type + "#" + metric_name] = get_stored_scores(df, lang_pair, context_type, model_name, metric_name)
        else:
          df[model_name + "#" + context_type + "#" + metric_name] = get_scores(df, model_name + "#" + context_type, metric_name)

    df.to_csv(f"analysis/{lang_pair}_ft_dev_with_scores_all.csv", index=None)

In [ ]:
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
sns.set_theme()
font = {'size'   : 30}
mpl.rc('font', **font)
mpl.rcParams['figure.dpi'] = 100


Contextual-comet-QE

In [ ]:
from typing import List
def read_file(fname):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    return output

# Referred from https://github.com/amazon-science/doc-mt-metrics/blob/main/Prism/add_context.py
def add_context(orig_txt: List[str], context_same: List[str], context_other: List[str], sender_ids: List[str], sep_token: str = "</s>", ws: int = 2) -> List[str]:
    if not (len(orig_txt) == len(context_same)== len(context_other)):
        raise Exception(f'Lengths should match: len(orig_txt)={len(orig_txt)}, len(context_same)={len(context_same)}, len(context_other)={len(context_other)}')
    i = 0
    augm_txt = []
    for i in range(len(orig_txt)):
      context_window = []
      for j in range(max(0, i - ws), i):
        if sender_ids[j] == sender_ids[i]:
          context_window.append(context_same[j])
        else:
          context_window.append(context_other[j])
      augm_txt.append(" {} ".format(sep_token).join(context_window + [orig_txt[i]]))
    return augm_txt

class DocCometMetric():
  def __init__(self, model_name="Unbabel/wmt20-comet-qe-da", batch_size=64, ref_based=True):
    checkpoint_path = download_model(model_name)
    self.model = load_from_checkpoint(checkpoint_path)
    self.batch_size = batch_size
    self.model.enable_context()
    self.ref_based = ref_based

  def get_score(self, source, outputs, references=None):
    if not self.ref_based:
      del references
      return self.model.predict([{"mt": y, "src": x} for x, y in zip(source, outputs)],
        batch_size=self.batch_size, gpus=1, progress_bar=True)['scores']
    else:
       return self.model.predict([{"mt": y, "ref":z, "src": x} for x, y, z in zip(source, outputs, references)],
        batch_size=self.batch_size, gpus=1, progress_bar=False, devices=[self.device_id])['scores']


In [ ]:
from comet import download_model, load_from_checkpoint
context_metric = DocCometMetric(model_name="Unbabel/wmt20-comet-qe-da", batch_size=256, ref_based=False)

In [ ]:
model_names = [
    # 'TowerInstruct-13B-v0.2_temperature_0.9_top_p_0.95',
    # 'TowerInstruct-13B-v0.2_temperature_0.8_top_p_0.9',
    # 'TowerInstruct-13B-v0.2_temperature_0.8_top_p_0.95',
    # 'TowerInstruct-13B-v0.2',
    # 'TowerInstruct-13B-v0.2_temperature_0.7_top_p_0.95',
    # 'TowerInstruct-13B-v0.2_temperature_0.6_top_p_0.95',
    # 'TowerInstruct-13B-v0.2_temperature_0.9_top_p_0.9',
    # 'TowerInstruct-13B-v0.2_temperature_1.0_top_p_0.9',
    # 'TowerInstruct-13B-v0.2_temperature_1.0_top_p_0.95',
    # 'TowerInstruct-13B-v0.2_temperature_0.6_top_p_0.9',
    # 'TowerInstruct-13B-v0.2_temperature_0.7_top_p_0.9',
    # 'TowerInstruct-7B-v0.2',
    'TowerInstruct-7B-w-chat-empty-sys',
                ]

In [ ]:
import pandas as pd

metric_name = "context-comet-qe"
for lang_pair in [ "en-de", "en-fr", "en-pt", "en-ko" ,  "en-nl",]:
    dfs_all = pd.read_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv")
    
    
    dfs_all = dfs_all.fillna('')
    context_size=2
    for model_name in model_names:
        for ctx in ["no_context_empty_sys", "full_context_empty_sys"]:
            doc_dfs = []
            for _, df_group in dfs_all.groupby(["doc_id"]):
                mt_col = model_name + "#" + ctx
                df_group['seg_id'] = list(range(len(df_group)))
                df_group[f"source_with_context"]  = add_context(
                                                        orig_txt=df_group["source"].to_list(),
                                                        context_same=df_group["source"].to_list(),
                                                        context_other=df_group[mt_col].to_list(),
                                                        sender_ids=df_group["sender"].to_list(),
                                                        sep_token=context_metric.model.encoder.tokenizer.sep_token,
                                                        ws=context_size,)
                df_group[f"mt_with_context"]  = add_context(
                                                        orig_txt=df_group[mt_col].to_list(),
                                                        context_same=df_group[mt_col].to_list(),
                                                        context_other=df_group["source"].to_list(),
                                                        sender_ids=df_group["sender"].to_list(),
                                                        sep_token=context_metric.model.encoder.tokenizer.sep_token,
                                                        ws=context_size,)
                doc_dfs.append(df_group)

            dfs_all = pd.concat(doc_dfs)
            
            dfs_all[mt_col + "#" + metric_name] =  context_metric.get_score(dfs_all[f"source_with_context"], dfs_all[f"mt_with_context"])
            dfs_all.to_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv", index=None)

# Analyis

## Can we select when to rely on context using QE metrics?

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd
import random
import numpy as np

def select_hypothesis(row, context_types, model_names, metric_name="comet-qe"):
    metric_scores = [row[f"{name}#{ctx}#{metric_name}"] for name in model_names for ctx in context_types]
    comet_scores = [row[f"{name}#{ctx}#comet"] for name in model_names for ctx in context_types] 
    chrf_scores = [row[f"{name}#{ctx}#chrf"] for name in model_names  for ctx in context_types]
    return comet_scores[np.argmax(metric_scores)],  chrf_scores[np.argmax(metric_scores)]


In [ ]:
# model_names = [
#                 'TowerInstruct-13B-v0.2_temperature_0.9_top_p_0.95',
#                 'TowerInstruct-13B-v0.2_temperature_0.8_top_p_0.9',
#                 'TowerInstruct-13B-v0.2_temperature_0.8_top_p_0.95',
#                 'TowerInstruct-13B-v0.2_temperature_0.7_top_p_0.95',
#                 'TowerInstruct-13B-v0.2_temperature_0.6_top_p_0.95',
#                 'TowerInstruct-13B-v0.2_temperature_0.9_top_p_0.9',
#                 'TowerInstruct-13B-v0.2_temperature_1.0_top_p_0.9',
#                 'TowerInstruct-13B-v0.2_temperature_1.0_top_p_0.95',
#                 'TowerInstruct-13B-v0.2_temperature_0.6_top_p_0.9',
#                 'TowerInstruct-13B-v0.2_temperature_0.7_top_p_0.9'
#                 ]

model_names=[]


In [ ]:
model_name='TowerInstruct-7B-w-chat-empty-sys'

full_context_key="full_context_empty_sys"
no_context_key="no_context_empty_sys"

for lang_pair in [ "en-de", "en-fr", "en-pt", "en-ko" ,  "en-nl",]:
    source_lang = lang_pair.split("-")[0]
    target_lang = lang_pair.split("-")[1].replace("pt", "pt-br")
    df = pd.read_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv")
    
    df[f"{no_context_key}#mean"] = df[[f"{model_name}#{no_context_key}#comet" for model_name in model_names+ [model_name]]].mean(axis=1)
    df[f"{full_context_key}#mean"] = df[[f"{model_name}#{full_context_key}#comet" for model_name in model_names+ [model_name]]].mean(axis=1)
    
    for metric in ["comet", "comet_kiwi", "comet-qe", "context-comet-qe"]:
        df[[f"{no_context_key}#{metric}_best#comet", f"{no_context_key}#{metric}_best#chrf"]] = df.apply(lambda x: select_hypothesis(x, [no_context_key],  model_names+ [model_name], metric), result_type='expand', axis=1)
        df[[f"{full_context_key}#{metric}_best#comet", f"{full_context_key}#{metric}_best#chrf"]] = df.apply(lambda x: select_hypothesis(x, [full_context_key],  model_names+ [model_name], metric), result_type='expand', axis=1)

        chrf_scores = []
        comet_scores = []
        for k in range(10):
            sampled_model_names = [model_name]
            # sampled_model_names = random.sample(model_names, 5) + [model_name]
            sample_comet, sample_chf = list(map(list, zip(*df.apply(lambda x: select_hypothesis(x, [no_context_key, full_context_key],  sampled_model_names, metric), axis=1).to_list())))
            comet_scores.append(sample_comet)
            chrf_scores.append(sample_chf)
        df[f"both#{metric}_best#comet"]= np.mean(np.array(comet_scores), axis=0)
        df[f"both#{metric}_best#chrf"] =  np.mean(np.array(chrf_scores), axis=0)
    
    df.to_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv", index=None)

In [ ]:
scores_list = []
for lang_pair in [ "en-de", "en-fr", "en-pt", "en-ko" ,  "en-nl",]:
    source_lang = lang_pair.split("-")[0]
    target_lang = lang_pair.split("-")[1].replace("pt", "pt-br")
    df_scores = pd.read_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv")
    for metric in ["comet", "comet_kiwi", "comet-qe", "context-comet-qe"]:
        
        for lp, lp_df in df_scores.groupby("lp"):
            if lp == f"{source_lang}-{target_lang}" or lp == f"{target_lang}-{source_lang}":
                scores_list.append([lp, metric, 
                                lp_df[f'{no_context_key}#{metric}_best#comet'].mean(),
                                lp_df[f'{full_context_key}#{metric}_best#comet'].mean(),
                                lp_df[f'both#{metric}_best#comet'].mean(),
                                lp_df[f'{no_context_key}#{metric}_best#chrf'].mean(),
                                lp_df[f'{full_context_key}#{metric}_best#chrf'].mean(),
                                lp_df[f'both#{metric}_best#chrf'].mean(),
                                ])

scores_list_df = pd.DataFrame(scores_list)
scores_list_df.columns = ["LP", "Metric", "No_Context (Comet)", "Full_Context (Comet)", "Both (Comet-S)","No_Context (Chrf)", "Full_Context (Chrf)", "Both (Chrf-S)"]

In [ ]:
gr_df = scores_list_df[scores_list_df["Metric"]=="comet"]

for ctx in ["No_Context",  "Full_Context"]:
    print("====")
    print(ctx)

    score_str = ("\t\t").join([ "en-de", "en-fr", "en-ko" ,  "en-nl", "en-pt-br"]) + "\n"
    for lang_pair in [ "en-de", "en-fr", "en-ko" ,  "en-nl", "en-pt-br"]:
        score_str+= f"{gr_df[gr_df['LP']==lang_pair][f'{ctx} (Chrf)'].values[0]:.4f}\t{gr_df[gr_df['LP']==lang_pair][f'{ctx} (Comet)'].values[0]:.4f}\t"
    print(score_str)

    score_str = ("\t\t").join([ "de-en", "fr-en", "ko-en" ,  "nl-en", "pt-br-en"]) + "\n"
    for lang_pair in [ "de-en", "fr-en", "ko-en" ,  "nl-en", "pt-br-en"]:
        score_str+= f"{gr_df[gr_df['LP']==lang_pair][f'{ctx} (Chrf)'].values[0]:.4f}\t{gr_df[gr_df['LP']==lang_pair][f'{ctx} (Comet)'].values[0]:.4f}\t"
    print(score_str)

    print("====\n")


In [ ]:
for gr, gr_df in scores_list_df.groupby("Metric"):
    print("====")
    print(gr)
    
    score_str = ("\t\t").join([ "en-de", "en-fr", "en-ko" ,  "en-nl", "en-pt-br"]) + "\n"
    for lang_pair in [ "en-de", "en-fr", "en-ko" ,  "en-nl", "en-pt-br"]:
        score_str+= f"{gr_df[gr_df['LP']==lang_pair]['Both (Chrf-S)'].values[0]:.4f}\t{gr_df[gr_df['LP']==lang_pair]['Both (Comet-S)'].values[0]:.4f}\t"
    print(score_str)

    score_str = ("\t\t").join([ "de-en", "fr-en", "ko-en" ,  "nl-en", "pt-br-en"]) + "\n"
    for lang_pair in [ "de-en", "fr-en", "ko-en" ,  "nl-en", "pt-br-en"]:
        score_str+= f"{gr_df[gr_df['LP']==lang_pair]['Both (Chrf-S)'].values[0]:.4f}\t{gr_df[gr_df['LP']==lang_pair]['Both (Comet-S)'].values[0]:.4f}\t"
    print(score_str)

    print("====\n")


## How often does context help?

In [ ]:
import pandas as pd

model_name='TowerInstruct-7B-w-chat-empty-sys'

full_context_key="full_context_empty_sys"
no_context_key="no_context_empty_sys"

print("LP Full-Context No-Context Diff")
            
count = []
for lang_pair in [ "en-de", "en-fr", "en-pt", "en-ko" ,  "en-nl",]:
    source_lang = lang_pair.split("-")[0]
    target_lang = lang_pair.split("-")[1].replace("pt", "pt-br")
    df = pd.read_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv")
    for lp, lp_df in df.groupby("lp"):
        if lp == f"{source_lang}-{target_lang}" or lp == f"{target_lang}-{source_lang}":
            count_better = len(lp_df[lp_df[f"{model_name}#{full_context_key}#comet"] > lp_df[f"{model_name}#{no_context_key}#comet"]] )
            count_same = len(lp_df[lp_df[f"{model_name}#{full_context_key}#comet"] == lp_df[f"{model_name}#{no_context_key}#comet"]] )
            count_worse = len(lp_df[lp_df[f"{model_name}#{full_context_key}#comet"] < lp_df[f"{model_name}#{no_context_key}#comet"]] )
            count.append([lp, f"{count_better/len(lp_df):.3f}",f"{count_worse/len(lp_df):.3f}",f"{count_same/len(lp_df):.3f}"])
            print(lp, lp_df[f"{model_name}#{full_context_key}#comet"].mean(), lp_df[f"{model_name}#{no_context_key}#comet"].mean(), (lp_df[f"{model_name}#{full_context_key}#comet"]- lp_df[f"{model_name}#{no_context_key}#comet"]).mean())
            
count_df = pd.DataFrame(count)
count_df.columns= ["lp", "%Better",  "%Same",  "%Worse"]

In [ ]:
count_df

In [ ]:
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
sns.set_theme()
# font = {'size'   : 20}
# mpl.rc('font', **font)
mpl.rcParams['figure.dpi'] = 100
sns.set(font_scale=1.8) 
# Set the positions and width for the bars
bar_width = 0.85
r = np.arange(len(count_df))

# Convert DataFrame columns to numpy arrays
better = np.array(count_df["%Better"].astype(float))
same = np.array(count_df["%Same"].astype(float))
worse = np.array(count_df["%Worse"].astype(float))

# Create the first bar (top), the second bar, and the third bar (bottom)
plt.figure(figsize=(12, 6))
plt.bar(r, worse, color='#FF817E', edgecolor='white', width=bar_width, label="%Worse")
plt.bar(r, same, bottom=worse, color='#557f2d', edgecolor='white', width=bar_width, label="%Same")
plt.bar(r, better, bottom=worse+same, color='#2d7f5e', edgecolor='white', width=bar_width, label="%Better")

for i in range(len(r)):
    plt.text(r[i], worse[i] / 2, f'{worse[i]}%', ha='center', va='center', color='white', fontsize=12)
    plt.text(r[i], worse[i] + same[i] / 2, f'{same[i]}%', ha='center', va='center', color='white', fontsize=12)
    plt.text(r[i], worse[i] + same[i] + better[i] / 2, f'{better[i]}%', ha='center', va='center', color='white', fontsize=12)


# Add labels
plt.xlabel('Lp', fontweight='bold')
plt.xticks(r, count_df['lp'])
plt.ylabel('Percentage', fontweight='bold')
plt.title('How often does context help?')

# Create legend & Show graphic
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))
plt.show()

## How accurate are QE metrics in detecting the good translations?

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

scores_dict = {}
model_name="TowerInstruct-7B-v0.2"
print("LP No_Context Full_Context Oracle QE Context-QE Both")
for lang_pair in [ "en-de", "en-fr", "en-pt", "en-ko" ,  "en-nl",]:
    source_lang = lang_pair.split("-")[0]
    target_lang = lang_pair.split("-")[1].replace("pt", "pt-br")
    df = pd.read_csv(f"analysis/{lang_pair}_dev_with_scores.csv")

    df[f"{model_name}#oracle_comet"] = df[[f"{model_name}#no_context#comet", f"{model_name}#full_context#comet"]].max(axis=1)

    context_useful = df[f"{model_name}#full_context#comet"] >= df[f'{model_name}#no_context#comet']

    kiwi_signal = df[f"{model_name}#full_context#comet_kiwi"] >= df[f"{model_name}#no_context#comet_kiwi"]
    kiwi_confusion = classification_report(kiwi_signal, context_useful)
    df[f"{model_name}#comet (kiwi_signal)"] =  kiwi_signal * df[f"{model_name}#full_context#comet"]  + (1-kiwi_signal)* df[f"{model_name}#no_context#comet"]
    print(kiwi_confusion)


    qe_signal = df[f"{model_name}#full_context#comet-qe"] >= df[f"{model_name}#no_context#comet-qe"]
    qe_confusion = classification_report(qe_signal, context_useful)
    df[f"{model_name}#comet (qe_signal)"] =  qe_signal * df[f"{model_name}#full_context#comet"]  + (1-qe_signal)* df[f"{model_name}#no_context#comet"]
    print(qe_confusion)

    context_qe_signal= df[f"{model_name}#full_context#context-comet-qe"] >= df[f"{model_name}#no_context#context-comet-qe"]
    context_qe_confusion = classification_report(context_qe_signal, context_useful)
    df[f"{model_name}#comet (context-qe_signal)"] =  context_qe_signal * df[f"{model_name}#full_context#comet"]  + (1-context_qe_signal)* df[f"{model_name}#no_context#comet"]
    print(context_qe_confusion)

    both_signal = qe_signal & context_qe_signal
    both_confusion = classification_report(both_signal, context_useful)
    df[f"{model_name}#comet (context-both_signal)"] =  both_signal * df[f"{model_name}#full_context#comet"] + (1-both_signal)* df[f"{model_name}#no_context#comet"]
    print(both_confusion)
    
    for lp, lp_df in df.groupby("lp"):
        if lp == f"{source_lang}-{target_lang}" or lp == f"{target_lang}-{source_lang}":
            print(f" {lp} "
                  f"{lp_df[f'{model_name}#no_context#comet'].mean():.4f} "
                  f"{lp_df[f'{model_name}#full_context#comet'].mean():.4f} "
                  f"{lp_df[f'{model_name}#oracle_comet'].mean():.4f}  "
                  f"{lp_df[f'{model_name}#comet (qe_signal)'].mean():.4f} "
                  f"{lp_df[f'{model_name}#comet (context-qe_signal)'].mean():.4f} "
                  f"{lp_df[f'{model_name}#comet (context-both_signal)'].mean():.4f} ")

In [ ]:
df.head()

# PXCMI

In [ ]:
def get_logprobs(df, lp, context_type, model_name="TowerInstruct-13B-v0.2", split="dev", prob_type="ref"):
  source_lang = lp.split("-")[0]
  target_lang = lp.split("-")[1].replace('pt','pt-br')

  xx_yy_logprobs = read_file(f"pcxmi/{context_type}/mt/wmt24_chat_{split}.{source_lang}-{target_lang}/{model_name}/mean_log_probs_{prob_type}.txt")
  yy_xx_logprobs = read_file(f"pcxmi/{context_type}/mt/wmt24_chat_{split}.{target_lang}-{source_lang}/{model_name}/mean_log_probs_{prob_type}.txt")

  assert len(xx_yy_logprobs) + len(yy_xx_logprobs) == len(df)
  translations = []
  j,k=0,0

  for i, row in df.iterrows():
    if row["lp"] == f"{source_lang}-{target_lang}":
      translations.append(xx_yy_logprobs[j])
      j+=1
    else:
      # nl-en and it-en are considered here
      translations.append(yy_xx_logprobs[k])
      k+=1
  return translations


In [ ]:
import pandas as pd

def add_segment_id(df):
    df_seg = []
    for _, gr_df in df.groupby("doc_id"):
        gr_df["seg_id"] = list(range(0, len(gr_df)))
        df_seg.append(gr_df)
    df = pd.concat(df_seg)
    return df

# model_name="TowerInstruct-7B-v0.2"

model_name='TowerInstruct-7B-w-chat-empty-sys'
full_context_key="full_context_empty_sys"
no_context_key="no_context_empty_sys"

for lang_pair in  [ "en-de", "en-fr", "en-pt", "en-ko" ,  "en-nl",]:
    source_lang = lang_pair.split("-")[0]
    target_lang = lang_pair.split("-")[1].replace("pt", "pt-br")
    df = pd.read_csv(f"chat-task-2024-data/valid/{lang_pair}.csv",
                  keep_default_na=False,
        na_values=["NaN"],)
    df["lp"] = df["source_language"] + "-" + df["target_language"]
    df["length"] = df["source"].apply(lambda x: len(x))
    
    for context_type in [no_context_key, full_context_key]:
      df[f"{model_name}#{context_type}#pcxmi (ref)"] = get_logprobs(df, lang_pair, context_type, model_name)
      df[f"{model_name}#{context_type}#pcxmi (hyp)"] = get_logprobs(df, lang_pair, context_type, model_name, prob_type="hyp")

    df = add_segment_id(df)
    df2 = pd.read_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv")
    df2 = add_segment_id(df2)

    df_pcxmi = pd.merge(df, df2,on=['doc_id', 'seg_id', 'source', 'reference', 'sender', 'lp', 'source_language', 'target_language', 'length', 'tags', 'client_id'], how='right')
    df_pcxmi.to_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv")
    

# Instructions

In [ ]:
def get_instructions(df, lp, context_type, split="dev"):
  source_lang = lp.split("-")[0]
  target_lang = lp.split("-")[1].replace('pt','pt-br')

  en_de_instructions = read_file(f"instructions/{context_type}/mt/wmt24_chat_{split}.{source_lang}-{target_lang}/instructions.txt")
  de_en_instructions = read_file(f"instructions/{context_type}/mt/wmt24_chat_{split}.{target_lang}-{source_lang}/instructions.txt")

  assert len(en_de_instructions) + len(de_en_instructions) == len(df)
  instructions = []
  j,k=0,0

  for i, row in df.iterrows():
    if row["lp"] == f"{source_lang}-{target_lang}":
      instructions.append(en_de_instructions[j])
      j+=1
    else:
      # nl-en and it-en are considered here
      instructions.append(de_en_instructions[k])
      k+=1
  return instructions

In [ ]:
import pandas as pd

model_name='TowerInstruct-7B-w-chat-empty-sys'
full_context_key="full_context_empty_sys"
no_context_key="no_context_empty_sys"

for lang_pair in  [ "en-de", "en-fr", "en-pt", "en-ko" ,  "en-nl",]:
    source_lang = lang_pair.split("-")[0]
    target_lang = lang_pair.split("-")[1].replace("pt", "pt-br")
    df = pd.read_csv(f"chat-task-2024-data/valid/{lang_pair}.csv",
                  keep_default_na=False,
        na_values=["NaN"],)
    df["lp"] = df["source_language"] + "-" + df["target_language"]
    df["length"] = df["source"].apply(lambda x: len(x))
    
    for context_type in [no_context_key, full_context_key]:
      df[f"{model_name}#{context_type}#instructions"] = get_instructions(df, lang_pair, context_type)
      
    df2 = pd.read_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv")

    df_inst = pd.merge(df, df2,on=['doc_id', 'source', 'reference', 'sender', 'lp', 'source_language', 'target_language', 'length', 'tags', 'client_id'], how='right')
    df_inst.drop(columns=["Unnamed: 0"], inplace=True)
    # df_inst.to_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv")
    

## When is context useful?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ast
import numpy as np

model_name='TowerInstruct-7B-w-chat-empty-sys'

full_context_key="full_context_empty_sys"
no_context_key="no_context_empty_sys"

def get_label(value):
    if value > 0:
        return 1
    elif value < 0:
        return -1
    else:
        return 0
            
count = []
dfs_all_list = []
for lang_pair in [ "en-de", "en-fr", "en-pt", "en-ko" ,  "en-nl",]:
    source_lang = lang_pair.split("-")[0]
    target_lang = lang_pair.split("-")[1].replace("pt", "pt-br")
    df = pd.read_csv(f"analysis/{lang_pair}_ft_dev_with_scores.csv")
    df["score_diff"] = df[f"{model_name}#{full_context_key}#comet"] - df[f"{model_name}#{no_context_key}#comet"]
    df["context helps?"] = df["score_diff"].apply(get_label)
    df['length binned'] = pd.cut(df['length'], [0, 20, 50, 100, 150])
    for metric in ["comet", "comet_kiwi", "comet-qe", "context-comet-qe"]:
        df[f'{metric} bucket'] = pd.cut(df[f"{model_name}#{no_context_key}#{metric}"], np.linspace(0, 1, 11))
    
    df['requires context'] = df['tags'].apply(lambda x: len(ast.literal_eval(x))!=0)
    # pcxmi (ref) -> mean logprob
    df["pcxmi (ref)"] = df[f"{model_name}#{full_context_key}#pcxmi (ref)"] - df[f"{model_name}#{no_context_key}#pcxmi (ref)"]
    df["pcxmi bins (ref)"] = pd.cut(df["pcxmi (ref)"], [-3.277975, -0.000352,  0.080790, 0.217844, 5.869184])

    df["pcxmi (hyp)"] = df[f"{model_name}#{full_context_key}#pcxmi (hyp)"] - df[f"{model_name}#{no_context_key}#pcxmi (hyp)"]
    df["pcxmi bins (hyp)"] = pd.cut(df["pcxmi (hyp)"], [-3.277975, -0.000352,  0.080790, 0.217844, 5.869184])

    dfs_all_list.append(df)

# dfs_all = pd.concat(dfs_all_list)

In [ ]:
threhold = 0.4

dfs_all = pd.concat(dfs_all_list)
dfs_all[f"kiwi_threshold-{threhold} (comet)"] = dfs_all.apply(lambda x: x[f"{model_name}#{no_context_key}#comet"] if  x[f"{model_name}#{no_context_key}#comet_kiwi"] < threhold else x[f"{model_name}#{full_context_key}#comet"], axis=1)
dfs_all[f"kiwi_threshold-{threhold} (chrf)"] = dfs_all.apply(lambda x: x[f"{model_name}#{no_context_key}#chrf"] if  x[f"{model_name}#{no_context_key}#comet_kiwi"] < threhold else x[f"{model_name}#{full_context_key}#chrf"], axis=1)

for lp, lp_df in dfs_all.groupby("lp"):
    for metric in ["comet", "chrf"]:
        print(lp, "kiwi-threhold", lp_df[f"kiwi_threshold-{threhold} ({metric})"].mean())
        print(lp, "no_context", lp_df[f"{model_name}#{no_context_key}#{metric}"].mean())
        print(lp, "full_context", lp_df[f"{model_name}#{full_context_key}#{metric}"].mean())
        print("\n")

In [ ]:
def plot_all(dfs_all, columns, normalized=True):
    for col in columns:    
        contingency_table = pd.crosstab(dfs_all[col], dfs_all['context helps?'])
        if normalized:
            normalized_contingency_table = contingency_table.div(contingency_table.sum(axis=1), axis=0)
            normalized_contingency_table.plot(kind='bar', stacked=True)
        else:
            contingency_table.plot(kind='bar', stacked=True)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.title(col)
        plt.show()
       
        contingency_table = df.groupby(col).agg({f"{model_name}#{full_context_key}#comet": 'mean', f"{model_name}#{no_context_key}#comet": 'mean'}).reset_index()
        contingency_table.columns = [col, "full_context", "no_context"]
        contingency_table = contingency_table.set_index(col)
        contingency_table.plot(kind='bar', stacked=False)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.title(col)
        plt.show()


In [ ]:
columns = ['length binned', 'seg_id', 'requires context', 'pcxmi bins (ref)', 'pcxmi bins (hyp)'] + [metric + " bucket" for metric in ["comet", "comet_kiwi", "comet-qe", "context-comet-qe"]]


In [ ]:
plot_all(dfs_all[dfs_all['source_language']=='en'], columns)

In [ ]:
dfs_all["comet_kiwi bucket"].unique()

In [ ]:
dfs_all[dfs_all["comet_kiwi bucket"]==dfs_all["comet_kiwi bucket"].unique()[4]]